In [1]:
%reset -f

##### Hardware initialization

In [2]:
# MCL Z positioner

from hardware.mclController import MCLStage, loadMCLDLL, ProductInformation
import time

print("Initializing Stage")
try: # if stage was already initialized, shut it down first
    stage.shutDown()
    time.sleep(0.5)
except NameError:
    pass

# DLL file location
mcl_lib = "c:/Program Files/Mad City Labs/NanoDrive/Madlib"  

stage = MCLStage(mcl_lib)
if not stage.getStatus():
    exit()

print("Stage Properties:")
stageProperties = stage.getProperties()
for key in sorted(stageProperties):
        print(key, '\t', stageProperties[key])
print("")

print("Z axis range:", "%.2f um" % stage.getAxisRange(3))
print("")

zPosition = 50 # initial z position in microns
stage.zMoveTo(zPosition)
time.sleep(0.05) # let it move, minimum wait for 0.05s

stage.getPosition(3)
print("Current Z Position:", "%.4f um" %stage.getPosition(3))
print("")

Initializing Stage
Stage Properties:
ADC_resolution 	 4096
DAC_resolution 	 256
FirmwareProfile 	 0
FirmwareVersion 	 6912
Product_id 	 544
SerialNumber 	 3846
axis_bitmap 	 4

Z axis range: 101.64 um

Current Z Position: 50.0344 um



In [3]:
# SPAD512 camera
# open the software first

from hardware.SPAD512S import SPAD512S
from pySPADutils import pySPADutils as util

port = 9990 # Check the command Server in the setting tab of the software and change it if necessary
SPAD1 = SPAD512S(port)

info = SPAD1.get_info()
print("\nGeneral informations of the camera :")
print(info)
temp = SPAD1.get_temps() # Current temperatures of FPGAs, PCB and Chip
print("\nCurrent temperatures of FPGAs, PCB and Chip :")
print(temp)
freq = SPAD1.get_freq() # Operating frequencies (Laser and frame)
print("\nOperating frequencies (Laser and frame) :")
print(freq)
SPAD1.enable_cooling(1) # Enable the cooling system of the camera


General informations of the camera :
['Master: 2433001CMT', 'Slave: 2433001CNR', 'Software version: 2.06', 'Firmware version: 56', 'Hardware version: 1', 'Hardware flavour: 512S SE', 'Hardware humidity sensor: 0', 'Intensity imaging: 1', 'Gated imaging: 1', 'FLIM imaging: 1']

Current temperatures of FPGAs, PCB and Chip :
('33.5', '34.3', '30.9', '24.1')

Operating frequencies (Laser and frame) :
['0.0', '0.0']


'Fans are already enabled.'

##### Acquization parameter setup

In [4]:
import numpy as np

# Camera settings
overlap = 1
timeout = 0
pileup = 1

im_width = 512
rowCol = 512 #Number of rows/columns
bitdepth = 1

# Each layer
BP = 100000
intTime = 10 # in us if 1-bit


# Z scan settings, 
# mode 0: single layer
# mode 1: Z stack, +/- Z_range around Z_middle, with step size of Z_stepSize


Z_mode = 0
Z_stepSize = 0.1    # in microns
Z_range    = 1     # in microns


Z_middle = 50 
match Z_mode:
    case 0:
        Z_positions = [Z_middle]
    case 1:
        n_steps = int(Z_range/Z_stepSize) + 1
        Z_positions = [Z_middle + (i-n_steps//2)*Z_stepSize for i in range(n_steps)]
    case 4951100:
        Z_positions = np.linspace(49, 51, 21)



print("Z positions to scan:", Z_positions, "microns")

zPosition = 50 # initial z position in microns
stage.zMoveTo(zPosition)
time.sleep(0.05) # let it move, minimum wait for 0.05s

stage.getPosition(3)
print("Current Z Position:", "%.4f um" %stage.getPosition(3))
print("")

Z positions to scan: [50] microns
Current Z Position: 49.9894 um



##### Exposure

In [24]:
from pytictoc import TicToc
from tqdm import tqdm
from pySPADutils import pySPADutils as util

import os
FOV = 5
os.mkdir("FOV_{}".format(FOV))

t = TicToc()
data = []
zbar = tqdm(Z_positions)
t.tic()
for z in zbar:
    zbar.set_description(f"Z @ {z:.2f} um")

    # Move stage to the z position
    stage.zMoveTo(z)
    time.sleep(0.05)
    
    # Acquire image at the z position
    data = SPAD1.get_intensity_1bit_packed(BP, intTime, bitdepth, overlap, timeout, pileup, im_width)
    filePath =  f"FOV_{FOV}/" + f"Z_{z*1000:.0f}.bin"
    util.writeBinBig(filePath, data)


t.toc("Acquisition time took")
zbar.close()

zPosition = 50 # initial z position in microns
stage.zMoveTo(zPosition)
time.sleep(0.05) # let it move, minimum wait for 0.05s

stage.getPosition(3)
print("Current Z Position:", "%.4f um" %stage.getPosition(3))
print("")

Z @ 50.00 um: 100%|██████████| 1/1 [00:54<00:00, 54.20s/it]

Acquisition time took 54.201195 seconds.
Current Z Position: 49.9884 um



##### Close stage handle

In [25]:
stage.shutDown()